# LLM Classification Finetuning — v0.0 EDA

Exploratory data analysis for the Chatbot Arena Human Preference Predictions competition.

**Key questions:**
1. Class balance — how skewed is A vs B vs tie?
2. Position bias — does response A win more often purely from position?
3. Verbosity bias — does the longer response win?
4. Length distributions — where should we set MAX_SEQ_LEN for truncation?
5. Missing values — any gaps in prompt / response fields?
6. Model identity — which LLMs appear in model_a / model_b (analysis only; never use as features).
7. Token budget — what fraction of examples exceed 512 / 1024 / 2048 / 4096 tokens?

No GPU required. CPU-only.

In [ ]:
# ── Cell 1: Install / imports ─────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'seaborn'], check=True)

import glob
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('Imports OK')

In [ ]:
# ── Cell 2: Load data ─────────────────────────────────────────────────────────
_candidates = [Path('/kaggle/input/llm-classification-finetuning')]
INPUT = next((p for p in _candidates if (p / 'train.csv').exists()), None)
if INPUT is None:
    _found = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    INPUT = Path(_found[0]).parent if _found else None
assert INPUT is not None, 'Competition data not found'

train = pd.read_csv(INPUT / 'train.csv')
test  = pd.read_csv(INPUT / 'test.csv')
sample = pd.read_csv(INPUT / 'sample_submission.csv')

winner_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
train['label'] = train[winner_cols].values.argmax(axis=1).astype(int)
train['label_str'] = train['label'].map({0: 'A', 1: 'B', 2: 'tie'})

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
print(f'Columns: {list(train.columns)}')
train.head(2)

In [ ]:
# ── Cell 3: Class balance & position bias ─────────────────────────────────────
counts = train['label_str'].value_counts().reindex(['A', 'B', 'tie'])
pct    = counts / len(train) * 100

print('=== Class balance ===')
for lbl, n, p in zip(counts.index, counts.values, pct.values):
    print(f'  {lbl:>4s}: {n:>6,}  ({p:.1f}%)')
print(f'  Total: {len(train):,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(counts.index, counts.values, color=['#4C72B0', '#DD8452', '#55A868'])
axes[0].set_title('Winner distribution')
axes[0].set_ylabel('Count')
for i, (n, p) in enumerate(zip(counts.values, pct.values)):
    axes[0].text(i, n + 50, f'{p:.1f}%', ha='center', va='bottom', fontsize=11)

# Position bias: if labels were random, A and B would each be ~50% of non-tie examples
non_tie = train[train['label_str'] != 'tie']
pos_bias = non_tie['label_str'].value_counts()
axes[1].bar(pos_bias.index, pos_bias.values, color=['#4C72B0', '#DD8452'])
axes[1].set_title('Position bias (non-tie only)\nExpected: equal A/B if no bias')
axes[1].set_ylabel('Count')
for i, (idx, n) in enumerate(pos_bias.items()):
    axes[1].text(i, n + 50, f'{n/len(non_tie)*100:.1f}%', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

bias = pos_bias.get('A', 0) / len(non_tie) - 0.5
print(f'\nPosition bias: A wins {pos_bias.get("A",0)/len(non_tie)*100:.1f}% of non-tie contests')
print(f'  Δ from 50%: {bias*100:+.1f}pp  ← swap augmentation targets this')

In [ ]:
# ── Cell 4: Length distributions (characters) ─────────────────────────────────
train['len_prompt'] = train['prompt'].fillna('').str.len()
train['len_a']      = train['response_a'].fillna('').str.len()
train['len_b']      = train['response_b'].fillna('').str.len()
train['len_total']  = train['len_prompt'] + train['len_a'] + train['len_b']

print('=== Character length stats ===')
for col in ['len_prompt', 'len_a', 'len_b', 'len_total']:
    q = train[col].quantile([0.5, 0.75, 0.9, 0.95, 0.99])
    print(f'  {col:<12s}  median={q[0.50]:>7,.0f}  p90={q[0.90]:>7,.0f}  p95={q[0.95]:>7,.0f}  p99={q[0.99]:>7,.0f}  max={train[col].max():>7,.0f}')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col, title in zip(
    axes.flat,
    ['len_prompt', 'len_a', 'len_b', 'len_total'],
    ['Prompt length (chars)', 'Response A length (chars)', 'Response B length (chars)', 'Total length (chars)']
):
    data = train[col].clip(upper=train[col].quantile(0.99))
    ax.hist(data, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    for q_pct, ls in [(0.50, '--'), (0.90, ':'), (0.95, '-')]:
        v = train[col].quantile(q_pct)
        ax.axvline(v, color='crimson', linestyle=ls, linewidth=1.2,
                   label=f'p{int(q_pct*100)}={v:,.0f}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5: Token budget (chars ÷ 4 approximation) ────────────────────────────
# Rule of thumb: ~4 chars per token for English text.
# Use this to plan MAX_SEQ_LEN without loading the tokenizer.
CHARS_PER_TOKEN = 4
train['tok_total'] = (train['len_total'] / CHARS_PER_TOKEN).astype(int)

thresholds = [512, 1024, 1536, 2048, 4096]
print('=== Estimated token budget coverage ===')
print(f'  {"MAX_SEQ_LEN":<12s}  examples covered  % of train')
for t in thresholds:
    covered = (train['tok_total'] <= t).sum()
    print(f'  {t:<12,}  {covered:>8,}          {covered/len(train)*100:.1f}%')

fig, ax = plt.subplots(figsize=(10, 4))
data = train['tok_total'].clip(upper=train['tok_total'].quantile(0.99))
ax.hist(data, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
for t in thresholds:
    ax.axvline(t, color='crimson', linewidth=1.2, linestyle='--', label=str(t))
ax.set_title('Estimated total token length (prompt + resp_a + resp_b)')
ax.set_xlabel('Tokens (approx, chars÷4)')
ax.set_ylabel('Count')
ax.legend(title='MAX_SEQ_LEN', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6: Verbosity bias — does the longer response win? ────────────────────
train['longer_wins'] = (
    ((train['len_a'] > train['len_b']) & (train['label_str'] == 'A')) |
    ((train['len_b'] > train['len_a']) & (train['label_str'] == 'B'))
)
train['shorter_wins'] = (
    ((train['len_a'] < train['len_b']) & (train['label_str'] == 'A')) |
    ((train['len_b'] < train['len_a']) & (train['label_str'] == 'B'))
)
non_tie = train[train['label_str'] != 'tie']
longer_rate  = non_tie['longer_wins'].mean()
shorter_rate = non_tie['shorter_wins'].mean()
equal_rate   = 1 - longer_rate - shorter_rate

print('=== Verbosity bias (non-tie examples) ===')
print(f'  Longer response wins : {longer_rate*100:.1f}%')
print(f'  Shorter response wins: {shorter_rate*100:.1f}%')
print(f'  Equal length         : {equal_rate*100:.1f}%')
print(f'  Baseline (random)    : 50.0%')
print(f'  Verbosity edge       : {(longer_rate - 0.5)*100:+.1f}pp')

# Length ratio distribution by winner
train['len_ratio_a_over_b'] = train['len_a'] / (train['len_b'] + 1)
fig, ax = plt.subplots(figsize=(10, 4))
for lbl, color in [('A', '#4C72B0'), ('B', '#DD8452'), ('tie', '#55A868')]:
    subset = train[train['label_str'] == lbl]['len_ratio_a_over_b'].clip(0, 5)
    ax.hist(subset, bins=60, alpha=0.5, label=f'Winner={lbl}', color=color, density=True)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.2, label='A=B length')
ax.set_title('Length ratio (len_a / len_b) by winner\nRight of 1.0 → A is longer')
ax.set_xlabel('len_a / len_b')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Missing values ────────────────────────────────────────────────────
print('=== Missing values — train ===')
missing = train[['prompt', 'response_a', 'response_b']].isnull().sum()
for col, n in missing.items():
    print(f'  {col:<12s}: {n:>5,}  ({n/len(train)*100:.2f}%)')

print('\n=== Missing values — test ===')
missing_test = test[['prompt', 'response_a', 'response_b']].isnull().sum()
for col, n in missing_test.items():
    print(f'  {col:<12s}: {n:>5,}  ({n/len(test)*100:.2f}%)')

# Empty strings (not NaN but effectively missing)
print('\n=== Empty strings — train ===')
for col in ['prompt', 'response_a', 'response_b']:
    n = (train[col].fillna('').str.strip() == '').sum()
    print(f'  {col:<12s}: {n:>5,}  ({n/len(train)*100:.2f}%)')

In [ ]:
# ── Cell 8: Model identity (analysis only — never use as features) ────────────
# model_a / model_b columns exist in train but NOT in test → leakage if used as features.
# Only use for CV stratification or understanding the data distribution.
model_cols = [c for c in train.columns if c.startswith('model_')]
if model_cols:
    print(f'Model identity columns found: {model_cols}')
    print('WARNING: These are absent from test.csv — do NOT use as model features (leakage).')
    print()
    for col in model_cols:
        vc = train[col].value_counts().head(20)
        print(f'=== {col} (top 20) ===')
        for model, n in vc.items():
            print(f'  {str(model):<50s} {n:>5,}')
        print()

    # Win rate by model (how often does each model win when it appears as model_a or model_b?)
    # Useful for understanding which models humans prefer
    all_models = pd.concat([
        train[['model_a', 'label_str']].rename(columns={'model_a': 'model', 'label_str': 'outcome'})
          .assign(side='a'),
        train[['model_b', 'label_str']].rename(columns={'model_b': 'model', 'label_str': 'outcome'})
          .assign(side='b'),
    ])
    all_models['won'] = (
        ((all_models['side'] == 'a') & (all_models['outcome'] == 'A')) |
        ((all_models['side'] == 'b') & (all_models['outcome'] == 'B'))
    )
    win_rates = (
        all_models.groupby('model')['won']
        .agg(['sum', 'count'])
        .assign(win_rate=lambda x: x['sum'] / x['count'])
        .query('count >= 100')
        .sort_values('win_rate', ascending=False)
        .head(20)
    )
    print('=== Win rate by model (min 100 appearances) ===')
    print(win_rates[['sum', 'count', 'win_rate']].rename(
        columns={'sum': 'wins', 'count': 'appearances', 'win_rate': 'win_rate'}
    ).to_string())
else:
    print('No model_a / model_b columns found in train.')

In [ ]:
# ── Cell 9: Duplicate / near-duplicate prompt analysis ────────────────────────
dup_prompts = train['prompt'].fillna('').duplicated(keep=False).sum()
dup_ids     = train['id'].duplicated().sum()

print(f'Duplicate prompt strings   : {dup_prompts:,}  ({dup_prompts/len(train)*100:.1f}%)')
print(f'Duplicate id values        : {dup_ids:,}')

# Prompt length by label — do tie examples have shorter / simpler prompts?
fig, ax = plt.subplots(figsize=(10, 4))
for lbl, color in [('A', '#4C72B0'), ('B', '#DD8452'), ('tie', '#55A868')]:
    subset = train[train['label_str'] == lbl]['len_prompt'].clip(
        upper=train['len_prompt'].quantile(0.99)
    )
    ax.hist(subset, bins=60, alpha=0.5, label=f'Winner={lbl}', color=color, density=True)
ax.set_title('Prompt length (chars) by winner')
ax.set_xlabel('Prompt length (chars)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: CV fold balance check ───────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('=== Stratified 5-fold label distribution ===')
for fold, (tr_idx, val_idx) in enumerate(skf.split(train, train['label'])):
    val_counts = train.iloc[val_idx]['label_str'].value_counts().reindex(['A','B','tie'])
    pcts = val_counts / val_counts.sum() * 100
    print(f'  Fold {fold}: A={pcts["A"]:.1f}%  B={pcts["B"]:.1f}%  tie={pcts["tie"]:.1f}%  n={len(val_idx):,}')

In [ ]:
# ── Cell 11: Summary of findings ─────────────────────────────────────────────
pos_bias_pp = (train[train['label_str']!='tie']['label_str'].value_counts(normalize=True).get('A', 0.5) - 0.5) * 100
verb_bias_pp = (longer_rate - 0.5) * 100

print('=' * 55)
print('FINDINGS SUMMARY')
print('=' * 55)
print(f'Train size          : {len(train):,}')
print(f'Test size           : {len(test):,}')
print()
print('Class balance:')
for lbl, p in zip(['A', 'B', 'tie'], pct.values):
    print(f'  {lbl:>4s}: {p:.1f}%')
print()
print(f'Position bias       : A wins {pos_bias_pp:+.1f}pp above 50% in non-tie examples')
print(f'Verbosity bias      : longer response wins {verb_bias_pp:+.1f}pp above 50% in non-tie')
print()
print('Token budget (approx chars÷4):')
for t in [1024, 1536, 2048]:
    cov = (train['tok_total'] <= t).mean() * 100
    print(f'  MAX_SEQ_LEN={t:<5}: covers {cov:.1f}% of examples')
print()
print('Modelling implications:')
print('  - Swap augmentation + swap-TTA is essential (position bias present)')
print('  - Length features worth including in DeBERTa / baseline (verbosity bias present)')
print('  - MAX_SEQ_LEN=1536 is a reasonable default; check coverage vs runtime')
if model_cols:
    print('  - model_a/model_b exist in train but NOT in test — never use as features')
print('=' * 55)